# Sacred Heart of Jesus Catholic School

Search OpenStreetMap for the school in **Boulder, Colorado**, then create an interactive map using the same libraries as `first-map.ipynb`.

In Codespaces, select the Python environment at `/opt/conda/bin/python` and choose **Run All**. The final cell saves `sacred-heart.html` in the current notebook working directory. The Haskell map is not overwritten.


In [ ]:
# Work with OpenStreetMap features and interactive maps.
from osmnx import features as osm
import holoviews as hv
import hvplot.pandas
from pathlib import Path

hv.extension('bokeh')


## Search for the school

OSMnx geocodes the full school name and city, then searches the surrounding 300 metres for features tagged `amenity=school`. The results can include other nearby schools, so we select Sacred Heart by name before plotting.

This uses **OSMnx**, not `pyrosm`. Internet access is required for the OpenStreetMap search and satellite basemap.


In [ ]:
school_query = (
    'Sacred Heart of Jesus Catholic School, '
    'Boulder, Colorado, United States'
)

nearby_schools = osm.features_from_address(
    school_query,
    tags={'amenity': 'school'},
    dist=300,
)

school_names = nearby_schools.get('name')
if school_names is None:
    raise ValueError('No named schools returned. Inspect nearby_schools before proceeding.')

school_gdf = nearby_schools.loc[
    school_names.fillna('').str.contains(
        'Sacred Heart of Jesus', case=False, regex=False
    )
].copy()

if school_gdf.empty:
    raise ValueError('Sacred Heart was not found among nearby schools. Inspect nearby_schools.')

# Inspect the selected feature before drawing the map.
columns = [column for column in ['name', 'amenity', 'geometry'] if column in school_gdf]
school_gdf[columns]


In [ ]:
school_gdf.plot()


## Create and save the interactive map

The outline comes from OpenStreetMap; satellite imagery is provided by Esri. Drag to pan and use the toolbar to zoom or reset the view.

Map data: [OpenStreetMap contributors](https://www.openstreetmap.org/copyright). School feature: [OSM way 749669576](https://www.openstreetmap.org/way/749669576).


In [ ]:
school_map = school_gdf.reset_index().hvplot(
    title='Sacred Heart of Jesus Catholic School, Boulder, CO',
    geo=True,
    tiles='EsriImagery',
    fill_color='white',
    fill_alpha=0.2,
    line_color='skyblue',
    line_width=5,
    width=550,
    height=500,
)

output_path = Path('sacred-heart.html')
hv.save(school_map, str(output_path))
print(f'Saved map: {output_path.resolve()}')
school_map


## Use the exported map

After the notebook finishes, download `sacred-heart.html` from the Codespace file explorer. To publish it later, put that file in your portfolio repository's `img` folder and embed `/img/sacred-heart.html` on the Demos page.

Stop the Codespace when finished so it does not continue using your allocation.
